In [ ]:
# Import pakietów i załadowanie danych

import pandas as pd
from bokeh.plotting import figure, output_file, show, ColumnDataSource, reset_output
from bokeh.io import show
from bokeh.layouts import column, row
from bokeh.models import Select, CustomJS, Slider, PreText, CheckboxGroup


df = pd.read_excel('2023_PM10_1g.xlsx')

In [1]:
# Filtracja danych

top_10_columns_sorted = df.isnull().sum().sort_values(ascending=True).head(10)
filtered_df = df[list(top_10_columns_sorted.index)]

NameError: name 'df' is not defined

In [ ]:
# Utworzenie wykresu z porownaniem danych dla 3 stacji wybierancyh z rozwijanej listy

TITLE = 'DataVisualizationPerStation'
output_file(TITLE+'.html')
source = ColumnDataSource(filtered_df)

p = figure(width=1500, height=800, x_axis_type='datetime', y_axis_label="[ug/m^3] PM10")

p.toolbar.logo = "grey"
p.background_fill_color = "#dddddd"
p.xaxis.axis_label = "DateTime"
p.grid.grid_line_color = "white"

p.line(x = "DateTime", y = "PkMielBierna", source = source, legend_label = "Stacja 1", color = "red" )
p.line(x = "DateTime", y = "KpToruWSikor", source = source, legend_label = "Stacja 2", color = "blue" )
p.line(x = "DateTime", y = "DsWrocWybCon", source = source, legend_label = "Stacja 3", color = "green" )

ys = Select(title= "Stacja 1", value = 'PkMielBierna', options = filtered_df.columns.tolist()[1:])
zs = Select(title= "Stacja 2", value = 'KpToruWSikor', options = filtered_df.columns.tolist()[1:])
ws = Select(title= "Stacja 3", value = 'DsWrocWybCon', options = filtered_df.columns.tolist()[1:])


callback1 = CustomJS(args=dict(source=source, sel1=ys), code=""" 
                     source.data['PkMielBierna'] = source.data[sel1.value];
                     
                     source.change.emit();
                     sel1.change.emit();
""")

callback2 = CustomJS(args=dict(source=source, sel2=zs), code=""" 
                     source.data['KpToruWSikor'] = source.data[sel2.value];
                     
                     source.change.emit();
                     sel2.change.emit();
""")

callback3 = CustomJS(args=dict(source=source, sel3=ws), code=""" 
                     source.data['DsWrocWybCon'] = source.data[sel3.value];
                     
                     source.change.emit();
                     sel3.change.emit();
""")

ys.js_on_change('value', callback1)
zs.js_on_change('value', callback2)
ws.js_on_change('value', callback3)
controls = column(ys, zs, ws )
layout = row(controls, p)

p.legend.click_policy='hide'

show(layout)

In [ ]:
# Przerobienie ramki danych na szeroka

filtered_df['Month'] = filtered_df['DateTime'].dt.month

stations = ["PkMielBierna", "KpToruWSikor", "DsWrocWybCon"]
reshaped_data = {}

for station in stations:
    for month in range(1, 13):
        col_name = f"{station}_Month{month}"
        reshaped_data[col_name] = filtered_df.loc[filtered_df['Month'] == month, station].reset_index(drop=True)

reshaped_df = pd.DataFrame(reshaped_data)

reshaped_df['DateTime'] = filtered_df['DateTime'].drop_duplicates().reset_index(drop=True)

for station in stations:
    reshaped_df[station] = filtered_df.loc[filtered_df['Month'] == 1, station].reset_index(drop=True)

C:\Users\Jakub\AppData\Local\Temp\ipykernel_11128\179948369.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Month'] = filtered_df['DateTime'].dt.month


In [ ]:
# Utworzenie wykresu z suwakiem sluzacym do wyboru miesiaca dla 3 rozpatrywanych stacji

output_file("DataVisualizationWithSlider.html")
default_month = 1
month_names = ["January", "February", "March", "April", "May", "June", 
               "July", "August", "September", "October", "November", "December"]

source = ColumnDataSource(reshaped_df)

p = figure(width=1500, height=800, x_axis_type='datetime', x_axis_label="Day", y_axis_label="[ug/m^3] PM10")
p.toolbar.logo = "grey"
p.background_fill_color = "#dddddd"
p.grid.grid_line_color = "white"

p.line(x="DateTime", y="PkMielBierna", source=source, legend_label="PkMielBierna", color="red", line_width=2)
p.line(x="DateTime", y="KpToruWSikor", source=source, legend_label="KpToruWSikor", color="blue", line_width=2)
p.line(x="DateTime", y="DsWrocWybCon", source=source, legend_label="DsWrocWybCon", color="green", line_width=2)

slider = Slider(start=1, end=12, value=default_month, step=1, title="Select Month: January", show_value = False)

callback = CustomJS(args=dict(source=source, original_data=reshaped_df), code="""
    const month = cb_obj.value;

    const col1 = `PkMielBierna_Month${month}`;
    const col2 = `KpToruWSikor_Month${month}`;
    const col3 = `DsWrocWybCon_Month${month}`;

    source.data['PkMielBierna'] = source.data[col1];
    source.data['KpToruWSikor'] = source.data[col2];
    source.data['DsWrocWybCon'] = source.data[col3];

    source.change.emit();
""");

callback2 = CustomJS(args=dict(slider=slider, month_names=month_names), code="""
    const month = cb_obj.value;
    const month_name = month_names[month - 1];  // Map month number to name
    slider.title = `Select Month: ${month_name}`;
""")

slider.js_on_change('value', callback)
slider.js_on_change('value', callback2)

p.xaxis.formatter.days = "%d"
p.xaxis.major_label_orientation = 0.5

layout = column(slider, p)

p.legend.click_policy = 'hide'

show(layout)